# Debug Notebook – Data Loading & Join Pipeline

Use this notebook to step through each stage of the pipeline individually,
inspect intermediate outputs, and diagnose any issues before running `main.ipynb`.

## 0 · Setup – paths and imports

In [ ]:
import os, sys

# Resolve the project root (one level up from debug/)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "code"))

CONFIG_PATH = os.path.join(PROJECT_ROOT, "config.json")

print(f"Project root : {PROJECT_ROOT}")
print(f"Config path  : {CONFIG_PATH}")
print(f"Config exists: {os.path.isfile(CONFIG_PATH)}")

## 1 · Load and inspect config

In [ ]:
from data_processing import load_config

config = load_config(CONFIG_PATH)

import json
print(json.dumps(config, indent=2))

In [ ]:
# Unpack config values
BASE_DIR          = config["global"]["base_data_dir"]
MASTER_FILE       = config["global"]["master_data"]
AUX_FILE          = config["global"]["aux_data_car_dep_fold"]
JOIN_KEY          = config["preprocess"]["joinkey"]
COMBINED_OUT      = config["preprocess"]["combined_output"]
QUALITY_DIR       = config["preprocess"]["quality_report_dir"]
QUALITY_FILE      = config["preprocess"]["quality_report_file"]

print("BASE_DIR     :", BASE_DIR)
print("MASTER_FILE  :", MASTER_FILE)
print("AUX_FILE     :", AUX_FILE)
print("JOIN_KEY     :", JOIN_KEY)
print("COMBINED_OUT :", COMBINED_OUT)
print("QUALITY_DIR  :", QUALITY_DIR)
print("QUALITY_FILE :", QUALITY_FILE)

## 2 · Verify data files exist on disk

In [ ]:
master_path = os.path.join(BASE_DIR, MASTER_FILE)
aux_path    = os.path.join(BASE_DIR, AUX_FILE)

print(f"Master path  : {master_path}")
print(f"Master exists: {os.path.isfile(master_path)}")
print()
print(f"Aux path     : {aux_path}")
print(f"Aux exists   : {os.path.isfile(aux_path)}")

## 3 · Load master_data and inspect

In [ ]:
from data_processing import load_master_data

master_df = load_master_data(BASE_DIR, MASTER_FILE)

print(f"Shape  : {master_df.shape}")
print(f"Columns: {list(master_df.columns)}")
print()
master_df.dtypes

In [ ]:
# Check join key presence
print(f"Join key '{JOIN_KEY}' in master_df: {JOIN_KEY in master_df.columns}")
print(f"Null count in join key: {master_df[JOIN_KEY].isna().sum()}")
print(f"Unique join-key values : {master_df[JOIN_KEY].nunique()}")
print()
master_df.head(5)

## 4 · Load aux_data and inspect

In [ ]:
from data_processing import load_aux_data

aux_df = load_aux_data(BASE_DIR, AUX_FILE)

print(f"Shape  : {aux_df.shape}")
print(f"Columns: {list(aux_df.columns)}")
print()
aux_df.dtypes

In [ ]:
print(f"Join key '{JOIN_KEY}' in aux_df: {JOIN_KEY in aux_df.columns}")
print(f"Null count in join key: {aux_df[JOIN_KEY].isna().sum()}")
print(f"Unique join-key values : {aux_df[JOIN_KEY].nunique()}")
print()
aux_df.head(5)

## 5 · Inspect join-key overlap before merging

In [ ]:
import pandas as pd

master_keys = set(master_df[JOIN_KEY].dropna().unique())
aux_keys    = set(aux_df[JOIN_KEY].dropna().unique())

common          = master_keys & aux_keys
only_in_master  = master_keys - aux_keys
only_in_aux     = aux_keys - master_keys

print(f"Common keys             : {len(common):,}")
print(f"Keys only in master     : {len(only_in_master):,}")
print(f"Keys only in aux        : {len(only_in_aux):,}")

if only_in_master:
    print("\nSample keys only in master (up to 10):")
    print(list(only_in_master)[:10])

if only_in_aux:
    print("\nSample keys only in aux (up to 10):")
    print(list(only_in_aux)[:10])

## 6 · Perform the inner join

In [ ]:
from data_processing import join_data

combined_df, drop_report = join_data(master_df, aux_df, JOIN_KEY, join_type="inner")

print("=== Drop Report ===")
for k, v in drop_report.items():
    print(f"  {k:<30}: {v}")

print()
print(f"Combined shape : {combined_df.shape}")
combined_df.head(5)

## 7 · Data quality check on the combined dataset

In [ ]:
from data_processing import check_data_quality

quality_df = check_data_quality(combined_df)

print("Columns with NaN:")
display(quality_df[quality_df["nan_count"] > 0])

print("\nNumeric columns with zeros:")
display(quality_df[quality_df["zero_count"].notna() & (quality_df["zero_count"] > 0)])

print("\nFull quality report:")
display(quality_df)

## 8 · (Optional) Inspect individual problem rows

In [ ]:
# Inspect rows with ANY NaN
nan_rows = combined_df[combined_df.isna().any(axis=1)]
print(f"Rows with at least one NaN: {len(nan_rows):,}")
nan_rows.head(10)

In [ ]:
# Inspect fully complete rows (no NaN anywhere)
complete_rows = combined_df.dropna()
print(f"Fully complete rows: {len(complete_rows):,} / {len(combined_df):,}")